# Continuation of 05_train_model.ipynb notebook
- Discovered contamination between model iterations.
- Clean new notebook for experimentation purely with transformer model.
- Reusable functions to ensure clean runs and development

# Load data and artefacts

In [12]:
# Imports
import io
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
import yaml
import pandas as pd
from minio import Minio

In [2]:
# Load config from yaml
CONFIG_FILE = Path("../config/config.yaml")

if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"Missing config file: {CONFIG_FILE.resolve()}")

with open(CONFIG_FILE, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

print("Config loaded")

Config loaded


In [3]:
# Read .env for credentials
def load_env_file(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing env file: {path.resolve()}")

    env = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        env[k.strip()] = v.strip()
    return env

# Env file location in repo
ENV_FILE = Path(f"{cfg['storage']['minio']['env_file']}")
# Storage endpoint and config
MINIO_ENDPOINT = cfg["storage"]["minio"]["endpoint"]
MINIO_SECURE = cfg["storage"]["minio"]["secure"]

# Authenticate to file storage list buckets to confirm connection
env = load_env_file(ENV_FILE)
client = Minio(
    MINIO_ENDPOINT,
    access_key=env["MINIO_ROOT_USER"],
    secret_key=env["MINIO_ROOT_PASSWORD"],
    secure=MINIO_SECURE,
)

print("Connected buckets:", [b.name for b in client.list_buckets()])

Connected buckets: ['data-profile-test', 'incident-pipeline', 'incident-pipeline-test', 'mlflow-artifacts']


In [4]:
# Load latest gold train, valid and test data from storage
gold_prefix_root = cfg["storage"]["minio"]["gold_training_prefix_root"].rstrip("/")
test_bucket = "incident-pipeline-test"

# Find latest timestamped gold run
run_tags = sorted({
    obj.object_name[len(gold_prefix_root) + 1:].split("/", 1)[0]
    for obj in client.list_objects(test_bucket, prefix=f"{gold_prefix_root}/", recursive=True)
    if obj.object_name.startswith(f"{gold_prefix_root}/") and len(obj.object_name.split("/")) >= 4
})

# Error if no gold runs found
if not run_tags:
    raise RuntimeError(f"No gold runs found under: {gold_prefix_root}")

# Use the latest gold run for training
latest_gold_run_tag = run_tags[-1]
latest_gold_run_prefix = f"{gold_prefix_root}/{latest_gold_run_tag}"

# Save this for model artifact traceability
training_data_run_tag = latest_gold_run_tag

# Paths for train, valid and test files
# Path for label mapping file
train_path = f"{latest_gold_run_prefix}/train.parquet"
valid_path = f"{latest_gold_run_prefix}/valid.parquet"
test_path = f"{latest_gold_run_prefix}/test.parquet"
label_mapping_path = f"{latest_gold_run_prefix}/label_mapping.json"

# Function to load parquet files
def load_parquet_from_storage(bucket: str, object_name: str) -> pd.DataFrame:
    resp = client.get_object(bucket, object_name)
    try:
        return pd.read_parquet(io.BytesIO(resp.read()))
    finally:
        resp.close()
        resp.release_conn()

# Function to load json files
def load_json_from_storage(bucket: str, object_name: str) -> dict:
    resp = client.get_object(bucket, object_name)
    try:
        return json.loads(resp.read().decode("utf-8"))
    finally:
        resp.close()
        resp.release_conn()

# Files to dfs
train_df = load_parquet_from_storage(test_bucket, train_path)
valid_df = load_parquet_from_storage(test_bucket, valid_path)
test_df = load_parquet_from_storage(test_bucket, test_path)
label_mapping = load_json_from_storage(test_bucket, label_mapping_path)

# Print summary info on the run and loaded data
print("Latest gold run tag:", latest_gold_run_tag)
print("Latest gold prefix:", latest_gold_run_prefix)
print("Train rows:", len(train_df))
print("Valid rows:", len(valid_df))
print("Test rows:", len(test_df))
print("Label mapping size:", len(label_mapping["classes"]))


Latest gold run tag: 20260308_205827
Latest gold prefix: gold/training/20260308_205827
Train rows: 7999
Valid rows: 1001
Test rows: 1000
Label mapping size: 12


In [5]:
# Label mapping check
display(label_mapping["classes"])

['App Support - ERP',
 'App Support - Finance',
 'App Support - HRIS',
 'App Support - M365',
 'App Support - Microsoft Fabric',
 'App Support - Power BI',
 'App Support - Power Platform',
 'End User Compute',
 'Identity and User Access',
 'Integration & Middleware',
 'Network Ops',
 'Security Operations']

# Validate data

In [6]:
# validate columns are present in all splits and check for leakage
# required cols
required_cols = ["sys_id", "text", "label_final", "label_id"]

# check for required cols in each split
for name, df in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")

    # Print now number, null rate, empty rate and number of labels in each split
    print(f"\n{name.upper()} checks")
    print("Rows:", len(df))
    print("Empty text rate:", f"{df['text'].fillna('').astype(str).str.strip().eq('').mean():.2%}")
    print("Null label rate:", f"{df['label_final'].isna().mean():.2%}")
    print("Unique labels:", df["label_final"].nunique())

# Collect sys_ids for leakage check
train_ids = set(train_df["sys_id"])
valid_ids = set(valid_df["sys_id"])
test_ids = set(test_df["sys_id"])

# Print overlap of sys_ids between splits to check for leakage
print("\nSplit leakage")
print("Train/Valid overlap:", len(train_ids & valid_ids))
print("Train/Test overlap :", len(train_ids & test_ids))
print("Valid/Test overlap :", len(valid_ids & test_ids))


TRAIN checks
Rows: 7999
Empty text rate: 0.00%
Null label rate: 0.00%
Unique labels: 12

VALID checks
Rows: 1001
Empty text rate: 0.00%
Null label rate: 0.00%
Unique labels: 12

TEST checks
Rows: 1000
Empty text rate: 0.00%
Null label rate: 0.00%
Unique labels: 12

Split leakage
Train/Valid overlap: 0
Train/Test overlap : 0
Valid/Test overlap : 0


# Functions for model development and iteration
Pattern for each model version:
- rebuild from the pretrained checkpoint
- retokenise the data
- create fresh training arguments
- train
- evaluate
- return the results in a consisent format

Development notes for each pre-processing step are available in the previous notebooks

In [ ]:
# Prepare data for transformer model
# Convert pandas splits into dataset objects
# Keep only requieed columns
def prepare_hf_datasets(train_df, valid_df, test_df):
    train_view = train_df[["sys_id", "text", "label_id", "label_final"]].copy()
    valid_view = valid_df[["sys_id", "text", "label_id", "label_final"]].copy()
    test_view = test_df[["sys_id", "text", "label_id", "label_final"]].copy()

    train_view = train_view.rename(columns={"label_id": "labels"})
    valid_view = valid_view.rename(columns={"label_id": "labels"})
    test_view = test_view.rename(columns={"label_id": "labels"})

    hf_train = Dataset.from_pandas(train_view, preserve_index=False)
    hf_valid = Dataset.from_pandas(valid_view, preserve_index=False)
    hf_test = Dataset.from_pandas(test_view, preserve_index=False)

    return hf_train, hf_valid, hf_test

In [ ]:
# Label mapping function to expected values
def prepare_label_maps(label_mapping):
    label_to_id = {k: int(v) for k, v in label_mapping["label_to_id"].items()}
    id_to_label = {int(k): v for k, v in label_mapping["id_to_label"].items()}
    label_names_in_order = [id_to_label[i] for i in sorted(id_to_label.keys())]

    return label_to_id, id_to_label, label_names_in_order

In [ ]:
# Build a fresh tokeniser and model from the original pretrained checkpoint
# Safe guard to prevent error that occured in previous notebook experiments
def build_tokeniser_and_model(checkpoint, label_mapping):
    label_to_id, id_to_label, _ = prepare_label_maps(label_mapping)

    tokeniser = AutoTokenizer.from_pretrained(checkpoint)

    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint,
        num_labels=len(label_mapping["classes"]),
        id2label=id_to_label,
        label2id=label_to_id,
    )

    return tokeniser, model

In [ ]:
# Tokenise all splits (train, test, val) and set max length value
def tokenise_splits(hf_train, hf_valid, hf_test, tokenizer, max_length):
    def tokenise_batch(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=max_length
        )

    tokenised_train = hf_train.map(tokenise_batch, batched=True)
    tokenised_valid = hf_valid.map(tokenise_batch, batched=True)
    tokenised_test = hf_test.map(tokenise_batch, batched=True)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    return tokenised_train, tokenised_valid, tokenised_test, data_collator

In [ ]:
# Metrics function to provide consistent evaluation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

In [ ]:
# Training arguments function. Default values as base model, fresh args per run with outputs timestamped
def build_training_args(
    run_name,
    output_root="./outputs",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    max_steps=-1,
    weight_decay=0.01,
    warmup_ratio=0.0,
    eval_strategy="no",
    save_strategy="no",
    load_best_model_at_end=False,
    metric_for_best_model=None,
    greater_is_better=True,
    logging_steps=20,
    seed=42,
    report_to="none",
):
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    output_dir = f"{output_root}/{run_name}_{timestamp}"

    training_args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=learning_rate,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=per_device_eval_batch_size,
        num_train_epochs=num_train_epochs,
        max_steps=max_steps,
        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,
        eval_strategy=eval_strategy,
        save_strategy=save_strategy,
        load_best_model_at_end=load_best_model_at_end,
        metric_for_best_model=metric_for_best_model,
        greater_is_better=greater_is_better,
        save_total_limit=1,
        seed=seed,
        logging_steps=logging_steps,
        report_to=report_to,
    )

    return training_args

In [ ]:
# Function to run an experiment end to end to centralise experiment flow and ensure a clean run each time.
# Reduces the risk of contamination between runs
def run_distilbert_experiment(
    run_name,
    hf_train,
    hf_valid,
    hf_test,
    label_mapping,
    checkpoint="distilbert/distilbert-base-uncased",
    max_length=128,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    max_steps=-1,
    weight_decay=0.01,
    warmup_ratio=0.0,
    eval_strategy="no",
    save_strategy="no",
    load_best_model_at_end=False,
    metric_for_best_model=None,
    greater_is_better=True,
    logging_steps=20,
    confidence_threshold=0.70,
):
    """
    Run one fully isolated DistilBERT experiment from a fresh checkpoint.
    """

    # Fresh model + tokenizer every run
    tokenizer, model = build_tokeniser_and_model(checkpoint, label_mapping)

    # Fresh tokenisation every run
    tokenized_train, tokenized_valid, tokenized_test, data_collator = tokenise_splits(
        hf_train, hf_valid, hf_test, tokenizer, max_length=max_length
    )

    # Fresh training args every run
    training_args = build_training_args(
        run_name=run_name,
        learning_rate=learning_rate,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=per_device_eval_batch_size,
        num_train_epochs=num_train_epochs,
        max_steps=max_steps,
        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,
        eval_strategy=eval_strategy,
        save_strategy=save_strategy,
        load_best_model_at_end=load_best_model_at_end,
        metric_for_best_model=metric_for_best_model,
        greater_is_better=greater_is_better,
        logging_steps=logging_steps,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_valid,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    start_time = time.time()
    trainer.train()
    elapsed_seconds = time.time() - start_time

    # Validation predictions
    valid_output = trainer.predict(tokenized_valid)
    valid_logits = valid_output.predictions
    valid_labels = valid_output.label_ids
    valid_preds = np.argmax(valid_logits, axis=-1)

    valid_accuracy = accuracy_score(valid_labels, valid_preds)
    valid_macro_f1 = f1_score(valid_labels, valid_preds, average="macro")

    # Confidence
    valid_probs = torch.softmax(torch.tensor(valid_logits), dim=1).numpy()
    max_probs = valid_probs.max(axis=1)
    manual_review_mask = max_probs < confidence_threshold
    manual_review_rate = float(manual_review_mask.mean())

    # Label outputs
    _, id_to_label, label_names_in_order = prepare_label_maps(label_mapping)

    class_report = classification_report(
        valid_labels,
        valid_preds,
        target_names=label_names_in_order,
        digits=2,
    )

    cm = confusion_matrix(
        valid_labels,
        valid_preds,
        labels=list(sorted(id_to_label.keys())),
    )

    cm_df = pd.DataFrame(
        cm,
        index=label_names_in_order,
        columns=label_names_in_order,
    )

    result = {
        "run_name": run_name,
        "checkpoint": checkpoint,
        "max_length": max_length,
        "learning_rate": learning_rate,
        "per_device_train_batch_size": per_device_train_batch_size,
        "per_device_eval_batch_size": per_device_eval_batch_size,
        "num_train_epochs": num_train_epochs,
        "max_steps": max_steps,
        "weight_decay": weight_decay,
        "warmup_ratio": warmup_ratio,
        "eval_strategy": eval_strategy,
        "save_strategy": save_strategy,
        "load_best_model_at_end": load_best_model_at_end,
        "global_steps": trainer.state.global_step,
        "training_time_seconds": elapsed_seconds,
        "validation_accuracy": float(valid_accuracy),
        "validation_macro_f1": float(valid_macro_f1),
        "avg_max_probability": float(max_probs.mean()),
        "manual_review_rate": manual_review_rate,
        "lowest_confidence": float(max_probs.min()),
        "highest_confidence": float(max_probs.max()),
        "classification_report": class_report,
        "confusion_matrix_df": cm_df,
        "output_dir": training_args.output_dir,
        "best_checkpoint": trainer.state.best_model_checkpoint,
    }

    return result

In [ ]:
# Function to print key outputs from an experiment run
def summarise_run(result):
    print(f"Run name: {result['run_name']}")
    print(f"Checkpoint: {result['checkpoint']}")
    print(f"Output dir: {result['output_dir']}")
    print(f"Global steps: {result['global_steps']}")
    print(f"Training time (mins): {result['training_time_seconds'] / 60:.2f}")
    print(f"Validation accuracy: {result['validation_accuracy']:.4f}")
    print(f"Validation macro F1: {result['validation_macro_f1']:.4f}")
    print(f"Average max probability: {result['avg_max_probability']:.4f}")
    print(f"Manual review rate @ 0.70: {result['manual_review_rate']:.4f}")
    print(f"Lowest confidence: {result['lowest_confidence']:.4f}")
    print(f"Highest confidence: {result['highest_confidence']:.4f}")

__________________________________________________________________________________________________________________________________________

In [13]:
# Prepare dataset once for consistency
hf_train, hf_valid, hf_test = prepare_hf_datasets(train_df, valid_df, test_df)

print("HF train rows:", len(hf_train))
print("HF valid rows:", len(hf_valid))
print("HF test rows :", len(hf_test))

NameError: name 'prepare_hf_datasets' is not defined

# Implementation candidate #2
DistilBERT challenger model

Candidate 1 saw good performance overall, especially on specialised classes that had consistent distinct wording. It did however struggle on broader classes that have overlap with one another in terms of the wording or terminology often used to describe problems. 

Candidate 2 should provide better performance on classes with overlaps as it utilises a transformer based model which is designed to handle these types of challenges. Unlike TF-IDF which mainly looks at word frequency and short combinations, DistilBERT is able to look at relationships between words in context. In theory, this should translate to a better understanding that two incidents can mean the same thing even if they are worded differently by the caller.

DistilBERT was the first choice due to it's smaller and faster nature when comapred with the full version of BERT, this should give us a good balance between model capability and computational cost.

At this stage, the model will be un-tuned, taking default configuration from hugging face before we iterate with optimisation. The goal is to establish whether the transformer model itself can provide a meaningful improvement over the baseline before spending time tuning it further.